# Factor Momentum on Equity Risk-Factor ETFs

This notebook implements and evaluates an ETF-based **cross-sectional Factor Momentum** strategy inspired by **Arnott et al. (2020)**.

## What the notebook covers
- **Gross and net backtests** with monthly rebalancing and intra-month weight drift
- **Historical daily risk-free adjustment** from **FRED** (default: `DGS3MO`)
- **Proxy transaction costs** based on daily **Abdi–Ranaldo (2017)** and **Corwin–Schultz (2012)** spread estimators
- **Estimator comparison** for net performance metrics and bootstrap summaries
- **Politis–Romano stationary-bootstrap** robustness analysis

## Practical note
The default configuration runs the full workflow, including bootstrap sections. For a quicker smoke test, set `RUN_BOOTSTRAP = False` and `RUN_ACF_BLOCK_SELECTION = False` in the configuration cell.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None


# ============================================================
# DATA
# ============================================================

TICKERS = {
    "Value": "RPV",
    "Size": "SIZE",
    "Momentum": "MTUM",
    "Quality": "SPHQ",
    "LowVol": "SPLV",
}

FACTOR_NAMES = ["Value", "Size", "Momentum", "Quality", "LowVol"]


def load_factor_etf_returns_hdf(filepath: str, key: str = "/FactorETFs") -> pd.DataFrame:
    """
    Load daily factor-ETF returns from an HDF store.
    Expected shape: 5 columns corresponding to Value / Size / Momentum / Quality / LowVol.
    """
    r = pd.read_hdf(filepath, key)
    r.index = pd.to_datetime(r.index)
    r.index.name = "Date"

    start = r.dropna().index[0]
    r = r.loc[start:].copy()

    if r.shape[1] == 5:
        r.columns = FACTOR_NAMES
    return r.astype(float)


def download_factor_etf_bundle(start: str = "2000-01-01") -> dict:
    """
    Download adjusted OHLC data for the factor ETF panel via Yahoo Finance.

    Returns
    -------
    dict with keys:
        close : adjusted daily close prices
        high  : adjusted daily highs
        low   : adjusted daily lows
        rets  : close-to-close returns from adjusted close
    """
    if yf is None:
        raise ImportError("yfinance is not available in this environment.")

    raw = yf.download(
        list(TICKERS.values()),
        start=start,
        auto_adjust=True,
        progress=False,
    )

    if not isinstance(raw.columns, pd.MultiIndex):
        raise ValueError("Expected a MultiIndex Yahoo Finance download for multiple tickers.")

    close = raw["Close"].rename(columns={v: k for k, v in TICKERS.items()})
    high = raw["High"].rename(columns={v: k for k, v in TICKERS.items()})
    low = raw["Low"].rename(columns={v: k for k, v in TICKERS.items()})

    close.index = pd.to_datetime(close.index)
    high.index = pd.to_datetime(high.index)
    low.index = pd.to_datetime(low.index)

    for obj in (close, high, low):
        obj.index.name = "Date"
        obj.sort_index(inplace=True)

    rets = close.pct_change().dropna(how="all")
    rets.index.name = "Date"

    return {
        "close": close.astype(float),
        "high": high.astype(float),
        "low": low.astype(float),
        "rets": rets.astype(float),
    }


# ============================================================
# HISTORICAL RISK-FREE RATE
# ============================================================

def fetch_fred_series(series_id: str, start: str | pd.Timestamp, end: str | pd.Timestamp) -> pd.Series:
    """
    Download a daily FRED series and forward-fill missing business-day observations.
    """
    if pdr is None:
        raise ImportError(
            "pandas_datareader is required to download FRED series. "
            "Install it or set USE_HISTORICAL_RF = False."
        )

    s = pdr.DataReader(series_id, "fred", start, end)
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]

    s.index = pd.to_datetime(s.index)
    s = s.astype(float).sort_index().ffill()
    s.name = series_id
    return s


def build_historical_rf_daily(
    index: pd.DatetimeIndex,
    rf_source: str = "DGS3MO",
    day_count: int = 360,
    lag_days: int = 1,
) -> pd.Series:
    """
    Build a historical daily simple risk-free return series aligned to `index`.

    Parameters
    ----------
    index      : target trading-calendar index
    rf_source  : FRED series id.
                 Recommended default for a full 2013+ sample: 'DGS3MO'.
                 'SOFR' is supported but only starts in 2018-04-03.
    day_count  : day-count convention used for the daily simple-rate approximation
    lag_days   : lag applied to avoid look-ahead bias

    Returns
    -------
    rf_daily : pd.Series of daily simple risk-free returns aligned to `index`
    """
    idx = pd.DatetimeIndex(index).sort_values().unique()
    if len(idx) == 0:
        return pd.Series(dtype=float, name="rf_daily")

    rf_source = rf_source.upper()
    if rf_source == "SOFR" and idx.min() < pd.Timestamp("2018-04-03"):
        raise ValueError(
            "SOFR starts on 2018-04-03. For a full 2013+ backtest sample, "
            "use RF_SOURCE='DGS3MO' (or truncate the sample to the SOFR era)."
        )

    start = idx.min() - pd.Timedelta(days=10)
    end = idx.max() + pd.Timedelta(days=10)

    y = fetch_fred_series(rf_source, start, end)

    rf_daily = (y / 100.0) / day_count
    rf_daily = rf_daily.shift(lag_days).ffill()

    rf = rf_daily.reindex(idx).ffill().bfill()
    rf.name = "rf_daily"
    return rf.astype(float)


In [ ]:
# ============================================================
# TRANSACTION-COST PROXIES FROM DAILY HLC DATA
# ============================================================

def abdi_ranaldo_spread(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    """
    Abdi–Ranaldo (2017) close-high-low estimator of the FULL proportional spread.
    A value of 0.01 corresponds to an estimated full spread of 1%.

    The daily estimate at date t uses:
      - close_{t-1}
      - eta_{t-1} = 0.5 * (log(high_{t-1}) + log(low_{t-1}))
      - eta_t     = 0.5 * (log(high_t)     + log(low_t))

    We clip negative estimates to zero, which is standard in practice.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)
    close = pd.Series(close).astype(float)

    idx = high.dropna().index.intersection(low.dropna().index).intersection(close.dropna().index)
    high = high.loc[idx].sort_index()
    low = low.loc[idx].sort_index()
    close = close.loc[idx].sort_index()

    log_h = np.log(high)
    log_l = np.log(low)
    log_c = np.log(close)

    eta = 0.5 * (log_h + log_l)
    s2 = 4.0 * (log_c.shift(1) - eta.shift(1)) * (log_c.shift(1) - eta)
    spread = np.sqrt(np.maximum(s2, 0.0))
    spread.name = "ar_spread"
    return spread


def abdi_ranaldo_panel(
    px_high: pd.DataFrame,
    px_low: pd.DataFrame,
    px_close: pd.DataFrame,
    tickers: list | None = None,
    clip_upper: float | None = 0.10,
) -> pd.DataFrame:
    """
    Apply Abdi–Ranaldo asset by asset on a panel of highs/lows/closes.
    Returns a DataFrame indexed by date, columns=tickers, with FULL spreads.
    """
    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns and c in px_close.columns]

    out = {}
    for t in tickers:
        h = px_high[t].dropna()
        l = px_low[t].dropna()
        c = px_close[t].dropna()

        idx = h.index.intersection(l.index).intersection(c.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue

        s = abdi_ranaldo_spread(h.loc[idx], l.loc[idx], c.loc[idx])

        if clip_upper is not None:
            s = s.clip(upper=clip_upper)

        out[t] = s

    panel = pd.DataFrame(out).sort_index()
    panel = panel.reindex(columns=tickers)
    return panel


# ============================================================
# Corwin–Schultz helpers (single asset + panel + one-way cost)
# ============================================================

def corwin_schultz_spread(high: pd.Series, low: pd.Series) -> pd.Series:
    """
    Corwin–Schultz (2012) bid–ask spread estimator from daily high/low prices.
    Returns the FULL proportional spread, e.g. 0.01 = 1%.
    Input must be two aligned Series for ONE asset.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)

    idx = high.dropna().index.intersection(low.dropna().index)
    high = high.loc[idx].sort_index()
    low = low.loc[idx].sort_index()

    hl = np.log(high / low)
    beta = hl.pow(2) + hl.shift(1).pow(2)

    high2 = pd.concat([high, high.shift(1)], axis=1).max(axis=1)
    low2 = pd.concat([low, low.shift(1)], axis=1).min(axis=1)
    gamma = np.log(high2 / low2).pow(2)

    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    alpha = alpha.clip(lower=0)

    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread.name = "cs_spread"
    return spread


def corwin_schultz_panel(
    px_high: pd.DataFrame,
    px_low: pd.DataFrame,
    tickers: list | None = None,
    clip_upper: float | None = 0.10,
) -> pd.DataFrame:
    """
    Apply Corwin–Schultz asset by asset on a panel of highs/lows.
    Returns a DataFrame indexed by date, columns=tickers, with FULL spreads.
    """
    px_high = px_high.copy()
    px_low = px_low.copy()

    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns]

    out = {}
    for t in tickers:
        h = px_high[t].dropna()
        l = px_low[t].dropna()

        idx = h.index.intersection(l.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue

        s = corwin_schultz_spread(h.loc[idx], l.loc[idx])

        if clip_upper is not None:
            s = s.clip(upper=clip_upper)

        out[t] = s

    panel = pd.DataFrame(out).sort_index()
    panel = panel.reindex(columns=tickers)
    return panel


def spread_panel_to_one_way_cost(spreads: pd.DataFrame) -> pd.DataFrame:
    """
    Convert FULL spread to ONE-WAY implementation cost.
    If spread = ask-bid over mid, a one-way execution cost is half-spread.
    """
    return 0.5 * spreads.astype(float)


def smooth_daily_cost_panel(
    cost_daily: pd.DataFrame,
    lookback_days: int = 21,
    agg: str = "median",
) -> pd.DataFrame:
    """
    Smooth noisy daily one-way costs with a trailing rolling window.
    This uses only information available up to each date.
    """
    cost_daily = cost_daily.sort_index().astype(float)

    min_obs = min(5, lookback_days)
    if agg == "median":
        smoothed = cost_daily.rolling(lookback_days, min_periods=min_obs).median()
    elif agg == "mean":
        smoothed = cost_daily.rolling(lookback_days, min_periods=min_obs).mean()
    else:
        raise ValueError("agg must be either 'median' or 'mean'")

    return smoothed


def align_costs_to_signal_dates(
    cost_daily: pd.DataFrame,
    signal_dates: pd.DatetimeIndex,
    lookback_days: int = 21,
    agg: str = "median",
) -> pd.DataFrame:
    """
    Build the per-asset one-way transaction-cost panel observed at month-end signal dates.
    The resulting panel can then be shifted to implementation dates.
    """
    cost_daily = cost_daily.sort_index().astype(float)
    smoothed = smooth_daily_cost_panel(cost_daily, lookback_days=lookback_days, agg=agg)
    out = smoothed.reindex(signal_dates)
    return out

In [ ]:

# ============================================================
# SIGNAL + WEIGHTS
# ============================================================

def month_end_trading_dates(idx: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """
    Last actual trading day of each month.
    """
    s = pd.Series(idx, index=idx)
    eom = s.groupby(s.index.to_period("M")).max()
    return pd.DatetimeIndex(eom.values)


def get_variant_params(variant: str) -> tuple[int, int]:
    """
    6-1  -> 6 months formation, 1 month skip
    12-1 -> 12 months formation, 1 month skip
    """
    tdpm = 21
    if variant == "6-1":
        return 6 * tdpm, 1 * tdpm
    elif variant == "12-1":
        return 12 * tdpm, 1 * tdpm
    raise ValueError("variant must be '6-1' or '12-1'")


def cs_mom_signal_monthly(
    r: pd.DataFrame,
    formation_days: int,
    skip_days: int,
) -> pd.DataFrame:
    """
    Cross-sectional momentum signal:
    signal_t = cumulative log-return over [t-skip-formation+1, ..., t-skip]
    computed daily, then sampled at month-end trading dates.
    """
    sig_daily = np.log1p(r).shift(skip_days).rolling(
        formation_days,
        min_periods=formation_days,
    ).sum()

    eom_dates = month_end_trading_dates(r.index)
    sig_m = sig_daily.loc[eom_dates].dropna(how="all")
    sig_m.index.name = "Date"
    return sig_m


def w_top1_80_20(sig_m: pd.DataFrame) -> pd.DataFrame:
    n = sig_m.shape[1]
    rest = 0.2 / (n - 1)
    w = pd.DataFrame(rest, index=sig_m.index, columns=sig_m.columns, dtype=float)
    top = sig_m.idxmax(axis=1)
    for dt, a in top.items():
        w.loc[dt, a] = 0.8
    return w


def w_top2_35_35(sig_m: pd.DataFrame) -> pd.DataFrame:
    n = sig_m.shape[1]
    rest = 0.3 / (n - 2)
    w = pd.DataFrame(rest, index=sig_m.index, columns=sig_m.columns, dtype=float)
    for dt in sig_m.index:
        top2 = sig_m.loc[dt].nlargest(2).index
        w.loc[dt, top2] = 0.35
    return w


def w_equal(sig_m: pd.DataFrame) -> pd.DataFrame:
    n = sig_m.shape[1]
    return pd.DataFrame(1.0 / n, index=sig_m.index, columns=sig_m.columns, dtype=float)


In [ ]:

# ============================================================
# BACKTEST
# ============================================================

def _entry_dates_from_signal_dates(
    trading_idx: pd.DatetimeIndex,
    signal_dates: pd.DatetimeIndex,
) -> tuple[pd.DatetimeIndex, pd.DatetimeIndex]:
    """
    Map each signal date (month-end close) to the next trading day,
    because weights can only be implemented after observing the signal.

    Signal dates prior to the first backtestable trading date are ignored.
    Duplicated entry dates are dropped later after scheduling.
    """
    trading_idx = pd.DatetimeIndex(trading_idx).sort_values().unique()
    signal_dates = pd.DatetimeIndex(signal_dates).sort_values().unique()

    entries = []
    keep = []
    for dt in signal_dates:
        if dt < trading_idx[0]:
            continue
        pos = trading_idx.searchsorted(dt, side="right")
        if pos < len(trading_idx):
            entries.append(trading_idx[pos])
            keep.append(dt)
    return pd.DatetimeIndex(entries), pd.DatetimeIndex(keep)


def backtest_monthly_rebalance_with_drift(
    r: pd.DataFrame,
    w_m: pd.DataFrame,
    cost_oneway_daily: pd.DataFrame | None = None,
    cost_lookback_days: int = 21,
    cost_agg: str = "median",
    charge_initial_trade: bool = True,
) -> tuple[pd.Series, pd.Series, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Correct monthly rebalance backtest with optional transaction costs.

    Important:
    - the signal is observed at month-end;
    - target weights are implemented on the NEXT trading day;
    - within the month, weights DRIFT with asset returns,
      i.e. there is no hidden daily rebalancing;
    - if cost_oneway_daily is provided, transaction costs are charged at each
      implementation date using drifted pre-trade weights and a one-way
      execution-cost panel.

    Returns
    -------
    port_gross : daily gross portfolio return series
    port_net   : daily net portfolio return series (equal to gross if costs are off)
    w_d        : daily start-of-day portfolio weights actually held
    w_sched    : target weights at each implementation date
    reb_info   : rebalance diagnostics (turnover, transaction-cost rate)
    """
    r = r.dropna(how="any").copy()

    entries, keep = _entry_dates_from_signal_dates(r.index, w_m.index)
    if len(entries) == 0:
        empty_s = pd.Series(dtype=float, name="portfolio")
        empty_w = pd.DataFrame(columns=r.columns, dtype=float)
        empty_info = pd.DataFrame(columns=["turnover", "tc_rate"], dtype=float)
        return empty_s, empty_s.copy(), empty_w, empty_w.copy(), empty_info

    w_sched = w_m.loc[keep].copy()
    w_sched.index = entries
    w_sched = w_sched[~w_sched.index.duplicated(keep="first")]
    entries = w_sched.index

    cost_sched = None
    if cost_oneway_daily is not None:
        cost_sig = align_costs_to_signal_dates(
            cost_oneway_daily.reindex(r.index),
            keep,
            lookback_days=cost_lookback_days,
            agg=cost_agg,
        )
        cost_sched = cost_sig.copy()
        cost_sched.index = entries
        cost_sched = cost_sched.reindex(index=entries, columns=r.columns).astype(float)

    port_gross = pd.Series(index=r.index, dtype=float, name="gross")
    port_net = pd.Series(index=r.index, dtype=float, name="net")
    w_d = pd.DataFrame(index=r.index, columns=r.columns, dtype=float)
    reb_info = pd.DataFrame(index=entries, columns=["turnover", "tc_rate"], dtype=float)

    entry_pos = [r.index.get_loc(dt) for dt in entries]
    current_w_pre = np.zeros(r.shape[1], dtype=float)

    for i, start_pos in enumerate(entry_pos):
        end_pos = entry_pos[i + 1] - 1 if i + 1 < len(entry_pos) else len(r.index) - 1

        target_w = w_sched.iloc[i].astype(float).values
        target_w = target_w / target_w.sum()

        if i == 0 and not charge_initial_trade:
            w_pre = target_w.copy()
        else:
            w_pre = current_w_pre.copy()

        turnover_rate = float(np.sum(np.abs(target_w - w_pre)))

        tc_rate = 0.0
        if cost_sched is not None:
            c_vec = cost_sched.iloc[i].fillna(0.0).astype(float).values
            tc_rate = float(np.sum(c_vec * np.abs(target_w - w_pre)))

        reb_info.iloc[i] = [turnover_rate, tc_rate]

        current_w = target_w.copy()

        for pos in range(start_pos, end_pos + 1):
            dt = r.index[pos]
            x = r.iloc[pos].astype(float).values

            w_d.loc[dt] = current_w

            gross_rp = float(np.dot(current_w, x))
            if pos == start_pos and tc_rate > 0:
                net_rp = (1.0 - tc_rate) * (1.0 + gross_rp) - 1.0
            else:
                net_rp = gross_rp

            port_gross.loc[dt] = gross_rp
            port_net.loc[dt] = net_rp

            current_w = current_w * (1.0 + x) / (1.0 + gross_rp)

        current_w_pre = current_w.copy()

    mask = port_gross.notna()
    return (
        port_gross.loc[mask],
        port_net.loc[mask],
        w_d.loc[mask],
        w_sched,
        reb_info,
    )


def _turnover_summary_from_rebalance_info(reb_info: pd.DataFrame) -> dict:
    if reb_info.empty:
        return {
            "n_rebalances": 0,
            "mean_turnover": np.nan,
            "median_turnover": np.nan,
            "annual_turnover_approx": np.nan,
            "mean_tc_rate": np.nan,
            "annual_tc_drag_approx": np.nan,
            "total_tc_rate": np.nan,
        }

    n_reb = int(len(reb_info))
    mean_to = float(reb_info["turnover"].mean())
    med_to = float(reb_info["turnover"].median())
    ann_to = 12.0 * mean_to
    mean_tc = float(reb_info["tc_rate"].mean())
    ann_tc = 12.0 * mean_tc
    total_tc = float(reb_info["tc_rate"].sum())

    return {
        "n_rebalances": n_reb,
        "mean_turnover": mean_to,
        "median_turnover": med_to,
        "annual_turnover_approx": ann_to,
        "mean_tc_rate": mean_tc,
        "annual_tc_drag_approx": ann_tc,
        "total_tc_rate": total_tc,
    }


def run_factor_mom_from_returns(
    r: pd.DataFrame,
    variant: str = "6-1",
    cost_oneway_daily: pd.DataFrame | None = None,
    cost_lookback_days: int = 21,
    cost_agg: str = "median",
    charge_initial_trade: bool = True,
) -> dict:
    """
    Run the full factor-momentum workflow on a common-data sample.

    Important: returns are first restricted to the common sample with no missing
    values across the factor ETF panel. Signals, schedules, transaction-cost
    alignment, and backtest dates are all constructed consistently on that
    common sample.
    """
    r_raw = r.copy()
    r = r.dropna(how="any").copy()

    formation_days, skip_days = get_variant_params(variant)
    sig_m = cs_mom_signal_monthly(
        r,
        formation_days=formation_days,
        skip_days=skip_days,
    )

    weight_schemes = {
        "MomTop1_80_20": w_top1_80_20(sig_m),
        "MomTop2_35_35": w_top2_35_35(sig_m),
        "EW": w_equal(sig_m),
    }

    if cost_oneway_daily is not None:
        cost_oneway_daily = cost_oneway_daily.reindex(r.index)

    rets_gross = {}
    rets_net = {}
    daily_weights = {}
    entry_weights = {}
    rebalance_info = {}
    turnover_summary = {}

    for name, w_m in weight_schemes.items():
        p_gross, p_net, w_d, w_sched, reb_info = backtest_monthly_rebalance_with_drift(
            r,
            w_m,
            cost_oneway_daily=cost_oneway_daily,
            cost_lookback_days=cost_lookback_days,
            cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade,
        )
        rets_gross[name] = p_gross.rename(name)
        rets_net[name] = p_net.rename(name)
        daily_weights[name] = w_d
        entry_weights[name] = w_sched
        rebalance_info[name] = reb_info
        turnover_summary[name] = _turnover_summary_from_rebalance_info(reb_info)

    rets_gross = pd.concat(rets_gross.values(), axis=1)
    rets_net = pd.concat(rets_net.values(), axis=1)

    avg_daily_weights = pd.concat({k: v.mean() for k, v in daily_weights.items()}, axis=1)
    avg_entry_weights = pd.concat({k: v.mean() for k, v in entry_weights.items()}, axis=1)
    turnover_summary = pd.DataFrame(turnover_summary).T

    sample_info = {
        "raw_start": r_raw.index.min(),
        "raw_end": r_raw.index.max(),
        "common_start": r.index.min(),
        "common_end": r.index.max(),
        "raw_rows": int(len(r_raw)),
        "common_rows": int(len(r)),
    }

    return {
        "rets_gross": rets_gross,
        "rets_net": rets_net,
        "signal": sig_m,
        "daily_weights": daily_weights,
        "entry_weights": entry_weights,
        "avg_daily_weights": avg_daily_weights,
        "avg_entry_weights": avg_entry_weights,
        "rebalance_info": rebalance_info,
        "turnover_summary": turnover_summary,
        "sample_info": sample_info,
    }


In [ ]:

# ============================================================
# PERFORMANCE METRICS
# ============================================================

def max_drawdown_from_wealth(wealth: pd.Series) -> float:
    wealth = wealth.dropna().astype(float)
    if len(wealth) == 0:
        return np.nan
    peak = wealth.cummax()
    dd = wealth / peak - 1.0
    return float(dd.min())


def ulcer_index(r: pd.Series) -> float:
    r = r.dropna().astype(float)
    if len(r) == 0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1.0
    return float(np.sqrt(np.mean(dd**2)))


def cagr(r: pd.Series, ann: int = 252) -> float:
    r = r.dropna().astype(float)
    if len(r) == 0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    years = len(r) / ann
    return float(wealth.iloc[-1] ** (1 / years) - 1) if years > 0 else np.nan


def _align_rf_to_returns(
    r: pd.Series,
    rf_daily: float | pd.Series | None = None,
) -> pd.Series:
    """
    Align a scalar or time-varying daily risk-free rate to the return series.
    """
    idx = r.dropna().index

    if rf_daily is None:
        return pd.Series(0.0, index=idx, name="rf_daily")

    if np.isscalar(rf_daily):
        return pd.Series(float(rf_daily), index=idx, name="rf_daily")

    if isinstance(rf_daily, pd.DataFrame):
        if rf_daily.shape[1] != 1:
            raise ValueError("rf_daily DataFrame must have exactly one column.")
        rf_daily = rf_daily.iloc[:, 0]

    rf = pd.Series(rf_daily).astype(float).sort_index()
    rf = rf.reindex(idx).ffill().bfill()
    rf.name = "rf_daily"
    return rf


def _annualized_rf_from_series(rf: pd.Series, ann: int = 252) -> float:
    rf = pd.Series(rf).dropna().astype(float)
    if len(rf) == 0:
        return 0.0
    gross = (1.0 + rf).prod()
    years = len(rf) / ann
    return float(gross ** (1 / years) - 1.0) if years > 0 else 0.0


def perf_metrics(
    r: pd.Series,
    rf_daily: float | pd.Series | None = None,
    ann: int = 252,
) -> dict:
    r = r.dropna().astype(float)
    if len(r) < 5:
        return {"n": int(len(r))}

    rf = _align_rf_to_returns(r, rf_daily=rf_daily)
    ex = r.loc[rf.index] - rf

    mu_d = ex.mean()
    vol_d = ex.std(ddof=1)
    shr = (mu_d / vol_d) * np.sqrt(ann) if vol_d > 0 else np.nan

    downside = ex.copy()
    downside[downside > 0] = 0.0
    dvol = downside.std(ddof=1)
    sor = (mu_d / dvol) * np.sqrt(ann) if dvol > 0 else np.nan

    cagr_ = cagr(r, ann=ann)
    ann_vol = r.std(ddof=1) * np.sqrt(ann)

    wealth = (1.0 + r).cumprod()
    mxdd = max_drawdown_from_wealth(wealth)
    calmar = (cagr_ / abs(mxdd)) if (mxdd < 0 and np.isfinite(mxdd)) else np.nan

    ui = ulcer_index(r)
    rf_ann = _annualized_rf_from_series(rf, ann=ann)
    martin = ((cagr_ - rf_ann) / ui) if (ui > 0 and np.isfinite(ui)) else np.nan

    return {
        "n": int(len(r)),
        "CAGR": float(cagr_),
        "AnnVol": float(ann_vol),
        "MxDD": float(mxdd),
        "ShR": float(shr),
        "SoR": float(sor),
        "Calmar": float(calmar),
        "Ulcer": float(ui),
        "Martin": float(martin),
    }


def perf_table(
    F: pd.DataFrame,
    rf_daily: float | pd.Series | None = None,
    ann: int = 252,
) -> pd.DataFrame:
    return pd.DataFrame(
        {c: perf_metrics(F[c], rf_daily=rf_daily, ann=ann) for c in F.columns}
    ).T


def wealth_index(F: pd.DataFrame) -> pd.DataFrame:
    return (1.0 + F).cumprod()


In [ ]:
# ============================================================
# POLITIS-ROMANO STATIONARY BOOTSTRAP
# ============================================================

def stationary_bootstrap_indices(
    T: int,
    avg_block_len: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    Politis-Romano stationary bootstrap:
    block lengths are geometric with mean avg_block_len.
    """
    p = 1.0 / avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)

    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t - 1] + 1) % T
    return idx


def stationary_bootstrap_df(
    df: pd.DataFrame,
    n_boot: int = 1000,
    avg_block_len: float = 10.0,
    seed: int = 1,
):
    """
    Bootstrap a daily panel and re-map the sample to the original trading calendar.
    """
    rng = np.random.default_rng(seed)
    T = len(df)

    for _ in range(n_boot):
        ii = stationary_bootstrap_indices(T, avg_block_len, rng)
        boot = df.iloc[ii].copy()
        boot.index = df.index
        yield boot


def stationary_bootstrap_joint_df(
    df_main: pd.DataFrame,
    df_aux: pd.DataFrame,
    n_boot: int = 1000,
    avg_block_len: float = 10.0,
    seed: int = 1,
):
    """
    Bootstrap two aligned daily panels jointly using the same Politis–Romano indices.
    Useful when returns and transaction-cost proxies should be resampled together.
    """
    rng = np.random.default_rng(seed)
    main = df_main.copy()
    aux = df_aux.reindex(df_main.index).copy()

    T = len(main)
    for _ in range(n_boot):
        ii = stationary_bootstrap_indices(T, avg_block_len, rng)

        boot_main = main.iloc[ii].copy()
        boot_aux = aux.iloc[ii].copy()

        boot_main.index = main.index
        boot_aux.index = main.index
        yield boot_main, boot_aux


def summarize_vec(x: np.ndarray) -> pd.Series:
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return pd.Series(
            {"mean": np.nan, "p05": np.nan, "p50": np.nan, "p95": np.nan, "n": 0}
        )

    return pd.Series({
        "mean": float(np.mean(x)),
        "p05": float(np.quantile(x, 0.05)),
        "p50": float(np.quantile(x, 0.50)),
        "p95": float(np.quantile(x, 0.95)),
        "n": int(len(x)),
    })


def _metric_tables_from_vectors(mats: dict, metrics: tuple, strategies: list) -> dict:
    out_tables = {}
    for m in metrics:
        rows = []
        for s in strategies:
            sm = summarize_vec(mats[(m, s)])
            sm["Strategy"] = s
            rows.append(sm)
        out_tables[m] = pd.DataFrame(rows).set_index("Strategy")
    return out_tables


def bootstrap_factor_mom_metrics(
    r: pd.DataFrame,
    variant: str = "6-1",
    cost_oneway_daily: pd.DataFrame | None = None,
    cost_lookback_days: int = 21,
    cost_agg: str = "median",
    charge_initial_trade: bool = True,
    metrics=("CAGR", "AnnVol", "ShR", "SoR", "Calmar", "Martin", "MxDD"),
    n_boot: int = 1000,
    avg_block_len: float = 10.0,
    seed: int = 1,
    rf_daily: float | pd.Series | None = None,
    ann: int = 252,
) -> dict:
    """
    Bootstrap the RAW factor-ETF returns and, inside each bootstrap replica,
    re-run the ENTIRE strategy construction:
      signal -> weights -> backtest -> metrics.

    If transaction costs are enabled, the aligned daily one-way cost panel is
    resampled jointly with returns using the same bootstrap indices.
    """
    actual = run_factor_mom_from_returns(
        r,
        variant=variant,
        cost_oneway_daily=cost_oneway_daily,
        cost_lookback_days=cost_lookback_days,
        cost_agg=cost_agg,
        charge_initial_trade=charge_initial_trade,
    )
    strategies = list(actual["rets_gross"].columns)

    actual_metrics_gross = perf_table(actual["rets_gross"], rf_daily=rf_daily, ann=ann)
    actual_metrics_net = perf_table(actual["rets_net"], rf_daily=rf_daily, ann=ann)

    mats_gross = {(m, s): np.full(n_boot, np.nan) for m in metrics for s in strategies}
    mats_net = {(m, s): np.full(n_boot, np.nan) for m in metrics for s in strategies}

    if cost_oneway_daily is None:
        boot_iter = (
            (sim_r, None)
            for sim_r in stationary_bootstrap_df(
                r,
                n_boot=n_boot,
                avg_block_len=avg_block_len,
                seed=seed,
            )
        )
    else:
        boot_iter = stationary_bootstrap_joint_df(
            r,
            cost_oneway_daily.reindex(r.index),
            n_boot=n_boot,
            avg_block_len=avg_block_len,
            seed=seed,
        )

    for i, (sim_r, sim_c) in enumerate(boot_iter):
        sim_out = run_factor_mom_from_returns(
            sim_r,
            variant=variant,
            cost_oneway_daily=sim_c,
            cost_lookback_days=cost_lookback_days,
            cost_agg=cost_agg,
            charge_initial_trade=charge_initial_trade,
        )
        tab_gross = perf_table(sim_out["rets_gross"], rf_daily=rf_daily, ann=ann)
        tab_net = perf_table(sim_out["rets_net"], rf_daily=rf_daily, ann=ann)

        for s in strategies:
            for m in metrics:
                mats_gross[(m, s)][i] = tab_gross.loc[s, m]
                mats_net[(m, s)][i] = tab_net.loc[s, m]

    return {
        "actual_metrics_gross": actual_metrics_gross,
        "actual_metrics_net": actual_metrics_net,
        "bootstrap_tables_gross": _metric_tables_from_vectors(mats_gross, metrics, strategies),
        "bootstrap_tables_net": _metric_tables_from_vectors(mats_net, metrics, strategies),
        "simulated_metric_vectors_gross": mats_gross,
        "simulated_metric_vectors_net": mats_net,
    }

In [ ]:
# ============================================================
# OPTIONAL: SIMPLE DATA-DRIVEN BLOCK-LENGTH CHOICE
# ============================================================

def _acf(x: np.ndarray, nlags: int) -> np.ndarray:
    x = np.asarray(x, float)
    x = x - np.mean(x)

    denom = np.dot(x, x)
    if denom <= 0:
        return np.zeros(nlags + 1)

    out = np.empty(nlags + 1, float)
    out[0] = 1.0
    for k in range(1, nlags + 1):
        out[k] = np.dot(x[:-k], x[k:]) / denom
    return out


def choose_block_length_by_acf_matching(
    s: pd.Series,
    candidates=(5, 10, 20, 30),
    nlags: int = 20,
    n_boot: int = 300,
    use_abs: bool = True,
    seed: int = 1,
    distance: str = "weighted",
) -> dict:
    """
    Pick block length by matching the ACF of the strategy (or abs(strategy)),
    which is useful when you want the bootstrap to preserve volatility clustering.
    """
    r = s.dropna().astype(float)
    x = np.abs(r.values) if use_abs else r.values
    target = _acf(x, nlags)

    rng = np.random.default_rng(seed)
    scores = {}
    boot_means = {}

    for L in candidates:
        acfs = np.zeros((n_boot, nlags + 1))

        for b in range(n_boot):
            ii = stationary_bootstrap_indices(len(x), float(L), rng)
            xb = x[ii]
            acfs[b] = _acf(xb, nlags)

        m = acfs.mean(axis=0)
        boot_means[L] = m
        diff = m[1:] - target[1:]  # ignore lag 0

        if distance == "l1":
            score = float(np.mean(np.abs(diff)))
        elif distance == "l2":
            score = float(np.sqrt(np.mean(diff**2)))
        elif distance == "weighted":
            w = 1.0 / np.arange(1, nlags + 1)
            score = float(np.sqrt(np.mean(diff**2 * w)))
        else:
            raise ValueError("distance must be 'l1', 'l2', or 'weighted'")

        scores[L] = score

    scores = pd.Series(scores).sort_index()

    return {
        "best_L": int(scores.idxmin()),
        "scores": scores,
        "target_acf": target,
        "boot_acf_mean": boot_means,
    }

## Configuration

Use the cell below to choose the data source and the main analysis options.

- `DATA_SOURCE = "hdf"` reproduces the original master-project return panel.
- `DATA_SOURCE = "yahoo"` uses a lightweight public-data workflow that is easier to share on GitHub.
- Transaction costs are implemented as **smoothed one-way half-spread proxies** applied at monthly rebalancing dates.
- Historical risk-free rates are pulled from **FRED** and aligned to the backtest dates.


In [ ]:

# ============================================================
# USER CONFIGURATION
# ============================================================

DATA_SOURCE = "yahoo"           # "yahoo" or "hdf"
HDF_PATH = None                 # e.g. "data/factor_etfs.h5" if DATA_SOURCE == "hdf"
HDF_KEY = "/FactorETFs"
START_DATE = "2000-01-01"
VARIANT = "6-1"                 # "6-1" or "12-1"

# --- Historical risk-free rate ---
USE_HISTORICAL_RF = True
RF_SOURCE = "DGS3MO"            # "DGS3MO", "DTB3", or "SOFR" (SOFR only from 2018-04-03 onward)
RF_DAY_COUNT = 360
RF_LAG_DAYS = 1

# --- Transaction costs ---
RUN_TRANSACTION_COSTS = True
TC_ESTIMATOR = "AR"             # primary estimator for plots/tables: "AR" or "CS"
RUN_TC_ESTIMATOR_COMPARISON = True
TC_ESTIMATORS_TO_COMPARE = ("AR", "CS")
TC_LOOKBACK_DAYS = 21
TC_AGG = "median"               # "median" or "mean"
TC_SPREAD_CLIP_UPPER = 0.10
TC_CHARGE_INITIAL_TRADE = True

# --- Bootstrap ---
RUN_ACF_BLOCK_SELECTION = True
ACF_CANDIDATES = (5, 10, 20, 30)
ACF_NLAGS = 20
ACF_N_BOOT = 300

RUN_BOOTSTRAP = True
N_BOOT = 1000
AVG_BLOCK_LEN = 10              # used when RUN_ACF_BLOCK_SELECTION = False
BOOTSTRAP_SEED = 7


## Load data, build the historical risk-free series, estimate transaction costs, and run the backtest

This section:
1. loads the raw factor-ETF return panel,
2. restricts the backtest to the **common-data sample**,
3. builds a **historical daily risk-free series** from **FRED**,
4. estimates daily transaction-cost panels under **Abdi–Ranaldo (AR)** and **Corwin–Schultz (CS)**,
5. runs the main backtest under the selected primary estimator, and
6. reports **gross** and **net** in-sample performance, turnover, and average portfolio weights.


In [ ]:

# ============================================================
# LOAD DATA
# ============================================================

bundle = None
if DATA_SOURCE == "hdf":
    if HDF_PATH is None:
        raise ValueError("Set HDF_PATH before using DATA_SOURCE='hdf'.")
    r_raw = load_factor_etf_returns_hdf(HDF_PATH, key=HDF_KEY)
    if RUN_TRANSACTION_COSTS:
        bundle = download_factor_etf_bundle(start=str(r_raw.index.min().date()))
elif DATA_SOURCE == "yahoo":
    bundle = download_factor_etf_bundle(start=START_DATE)
    r_raw = bundle["rets"]
else:
    raise ValueError("DATA_SOURCE must be either 'hdf' or 'yahoo'.")

r_raw = r_raw.copy()
r_common = r_raw.dropna(how="any").copy()

print(f"Loaded raw returns panel with shape {r_raw.shape} from {r_raw.index.min().date()} to {r_raw.index.max().date()}.")
display(r_raw.head())

print(f"Common-data backtest sample: {r_common.shape} from {r_common.index.min().date()} to {r_common.index.max().date()}.")

# ============================================================
# HISTORICAL RISK-FREE SERIES
# ============================================================

if USE_HISTORICAL_RF:
    rf_daily = build_historical_rf_daily(
        r_common.index,
        rf_source=RF_SOURCE,
        day_count=RF_DAY_COUNT,
        lag_days=RF_LAG_DAYS,
    )
    print(f"Historical risk-free source: {RF_SOURCE}")
    print("Daily risk-free sample (aligned to the common-data backtest index):")
    display(rf_daily.to_frame("rf_daily").head())
else:
    rf_daily = pd.Series(0.0, index=r_common.index, name="rf_daily")
    print("Historical risk-free disabled: using rf_daily = 0.0.")

# ============================================================
# ESTIMATE DAILY TRANSACTION-COST PANELS
# ============================================================

cost_oneway_daily_main = None
spread_full_panels = {}
cost_oneway_panels = {}
tc_mean_table = None
tc_corr_by_etf = None
tc_corr_overall = np.nan

estimators_requested = [TC_ESTIMATOR]
if RUN_TC_ESTIMATOR_COMPARISON:
    estimators_requested = list(dict.fromkeys([TC_ESTIMATOR] + list(TC_ESTIMATORS_TO_COMPARE)))

if RUN_TRANSACTION_COSTS:
    if bundle is None:
        bundle = download_factor_etf_bundle(start=str(r_raw.index.min().date()))

    px_close = bundle["close"].reindex(r_common.index)
    px_high = bundle["high"].reindex(r_common.index)
    px_low = bundle["low"].reindex(r_common.index)

    for est in estimators_requested:
        est = est.upper()
        if est == "AR":
            spread_full = abdi_ranaldo_panel(
                px_high,
                px_low,
                px_close,
                tickers=FACTOR_NAMES,
                clip_upper=TC_SPREAD_CLIP_UPPER,
            )
        elif est == "CS":
            spread_full = corwin_schultz_panel(
                px_high,
                px_low,
                tickers=FACTOR_NAMES,
                clip_upper=TC_SPREAD_CLIP_UPPER,
            )
        else:
            raise ValueError("Each transaction-cost estimator must be either 'AR' or 'CS'.")

        spread_full_panels[est] = spread_full.reindex(r_common.index)
        cost_oneway_panels[est] = spread_panel_to_one_way_cost(spread_full).reindex(r_common.index)

    cost_oneway_daily_main = cost_oneway_panels[TC_ESTIMATOR.upper()]

    print(f"Primary transaction-cost estimator: {TC_ESTIMATOR.upper()}")

    tc_mean_table = pd.concat(
        {
            est: cost_oneway_panels[est].mean().rename("one_way_cost")
            for est in cost_oneway_panels
        },
        axis=1,
    )
    print("Mean estimated one-way cost by ETF:")
    display(tc_mean_table.round(5))

    if {"AR", "CS"}.issubset(set(spread_full_panels.keys())):
        aligned_main = spread_full_panels["AR"].stack()
        aligned_alt = spread_full_panels["CS"].stack()
        tc_corr_overall = float(aligned_main.corr(aligned_alt))

        tc_corr_by_etf = pd.Series(
            {
                fac: float(spread_full_panels["AR"][fac].corr(spread_full_panels["CS"][fac]))
                for fac in FACTOR_NAMES
            },
            name="AR_vs_CS_corr",
        ).to_frame()

        print("Robustness check — correlation between AR and CS full-spread estimates:")
        print(round(tc_corr_overall, 4))
        print("By ETF:")
        display(tc_corr_by_etf.round(4))

# ============================================================
# RUN PRIMARY STRATEGY
# ============================================================

out = run_factor_mom_from_returns(
    r_raw,
    variant=VARIANT,
    cost_oneway_daily=cost_oneway_daily_main,
    cost_lookback_days=TC_LOOKBACK_DAYS,
    cost_agg=TC_AGG,
    charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
)

rets_gross = out["rets_gross"]
rets_net = out["rets_net"]
signal = out["signal"]
avg_daily_w = out["avg_daily_weights"]
avg_entry_w = out["avg_entry_weights"]
turnover_summary = out["turnover_summary"]
sample_info = out["sample_info"]

actual_metrics_gross = perf_table(rets_gross, rf_daily=rf_daily)
actual_metrics_net = perf_table(rets_net, rf_daily=rf_daily)

print("Backtest sample information:")
display(pd.Series(sample_info, name="value").to_frame())

print("Gross performance metrics:")
display(actual_metrics_gross.round(4))

print(f"Net performance metrics — primary estimator ({TC_ESTIMATOR.upper()}):")
display(actual_metrics_net.round(4))

print("Rebalancing / turnover summary:")
display(turnover_summary.round(4))

print("Average daily weights:")
display(avg_daily_w.round(4))

print("Average entry weights:")
display(avg_entry_w.round(4))

# ============================================================
# IN-SAMPLE NET COMPARISON: AR VS CS
# ============================================================

comparison_out = {}
comparison_metrics_net = {}
comparison_turnover = {}

if RUN_TRANSACTION_COSTS and RUN_TC_ESTIMATOR_COMPARISON:
    for est in estimators_requested:
        est_out = run_factor_mom_from_returns(
            r_raw,
            variant=VARIANT,
            cost_oneway_daily=cost_oneway_panels[est],
            cost_lookback_days=TC_LOOKBACK_DAYS,
            cost_agg=TC_AGG,
            charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
        )
        comparison_out[est] = est_out
        comparison_metrics_net[est] = perf_table(est_out["rets_net"], rf_daily=rf_daily)
        comparison_turnover[est] = est_out["turnover_summary"]

    print("Net in-sample performance metrics — estimator comparison (AR vs CS):")
    display(pd.concat(comparison_metrics_net, names=["Estimator", "Strategy"]).round(4))

    print("Turnover / transaction-cost summary — estimator comparison (AR vs CS):")
    display(pd.concat(comparison_turnover, names=["Estimator"]).round(4))


## Plot cumulative wealth (gross vs net)

In [ ]:
wealth_gross = wealth_index(rets_gross)
wealth_net = wealth_index(rets_net)

ax = wealth_gross.plot(figsize=(11, 5), title=f"Factor Momentum strategies ({VARIANT}) — gross")
ax.set_ylabel("Wealth index")
plt.show()

ax = wealth_net.plot(figsize=(11, 5), title=f"Factor Momentum strategies ({VARIANT}) — net")
ax.set_ylabel("Wealth index")
plt.show()

## Optional data-driven block-length selection

A simple heuristic is provided to choose the average block length by matching the autocorrelation function of **absolute returns** for the reference strategy. For this project, `MomTop1_80_20` is a sensible default.

When transaction costs are enabled, the selection is still performed on the **gross** strategy return series, while the bootstrap itself re-samples the aligned transaction-cost panel jointly with raw returns.


In [ ]:
if RUN_ACF_BLOCK_SELECTION:
    bl = choose_block_length_by_acf_matching(
        rets_gross["MomTop1_80_20"],
        candidates=ACF_CANDIDATES,
        nlags=ACF_NLAGS,
        n_boot=ACF_N_BOOT,
        use_abs=True,
        seed=BOOTSTRAP_SEED,
        distance="weighted",
    )
    best_L = bl["best_L"]
    print("Suggested avg_block_len:", best_L)
    display(bl["scores"])
else:
    best_L = AVG_BLOCK_LEN
    print("Using fixed avg_block_len:", best_L)

## Politis–Romano stationary bootstrap

The bootstrap re-samples the **raw factor-ETF return panel** and, when transaction costs are enabled, the **transaction-cost panel jointly with returns** using stationary-bootstrap blocks.

For each bootstrap replica, the notebook re-runs:
- momentum signal construction,
- portfolio-weight scheduling,
- gross and net backtests, and
- performance metrics under the same conventions used in-sample.

The final tables therefore reflect **strategy-level uncertainty after implementation frictions**, not just uncertainty in an already-computed return series.


In [ ]:

if RUN_BOOTSTRAP:
    # Primary bootstrap (gross + net under the selected primary estimator)
    boot_primary = bootstrap_factor_mom_metrics(
        r_raw,
        variant=VARIANT,
        cost_oneway_daily=cost_oneway_daily_main,
        cost_lookback_days=TC_LOOKBACK_DAYS,
        cost_agg=TC_AGG,
        charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
        n_boot=N_BOOT,
        avg_block_len=best_L,
        seed=BOOTSTRAP_SEED,
        rf_daily=rf_daily,
    )

    boot_actual_metrics_gross = boot_primary["actual_metrics_gross"]
    boot_actual_metrics_net = boot_primary["actual_metrics_net"]

    boot_cagr_gross = boot_primary["bootstrap_tables_gross"]["CAGR"]
    boot_shr_gross = boot_primary["bootstrap_tables_gross"]["ShR"]
    boot_martin_gross = boot_primary["bootstrap_tables_gross"]["Martin"]
    boot_mxdd_gross = boot_primary["bootstrap_tables_gross"]["MxDD"]

    boot_cagr_net = boot_primary["bootstrap_tables_net"]["CAGR"]
    boot_shr_net = boot_primary["bootstrap_tables_net"]["ShR"]
    boot_martin_net = boot_primary["bootstrap_tables_net"]["Martin"]
    boot_mxdd_net = boot_primary["bootstrap_tables_net"]["MxDD"]

    print("Actual backtest metrics — gross")
    display(boot_actual_metrics_gross.round(4))

    print(f"Actual backtest metrics — net ({TC_ESTIMATOR.upper()})")
    display(boot_actual_metrics_net.round(4))

    print("Bootstrap distribution — gross CAGR")
    display(boot_cagr_gross.round(4))

    print(f"Bootstrap distribution — net CAGR ({TC_ESTIMATOR.upper()})")
    display(boot_cagr_net.round(4))

    print("Bootstrap distribution — gross Sharpe ratio")
    display(boot_shr_gross.round(4))

    print(f"Bootstrap distribution — net Sharpe ratio ({TC_ESTIMATOR.upper()})")
    display(boot_shr_net.round(4))

    print("Bootstrap distribution — gross Martin ratio")
    display(boot_martin_gross.round(4))

    print(f"Bootstrap distribution — net Martin ratio ({TC_ESTIMATOR.upper()})")
    display(boot_martin_net.round(4))

    print("Bootstrap distribution — gross Max drawdown")
    display(boot_mxdd_gross.round(4))

    print(f"Bootstrap distribution — net Max drawdown ({TC_ESTIMATOR.upper()})")
    display(boot_mxdd_net.round(4))

    # AR vs CS comparison for net metrics
    if RUN_TRANSACTION_COSTS and RUN_TC_ESTIMATOR_COMPARISON:
        boot_comp = {}
        for est in estimators_requested:
            boot_comp[est] = bootstrap_factor_mom_metrics(
                r_raw,
                variant=VARIANT,
                cost_oneway_daily=cost_oneway_panels[est],
                cost_lookback_days=TC_LOOKBACK_DAYS,
                cost_agg=TC_AGG,
                charge_initial_trade=TC_CHARGE_INITIAL_TRADE,
                n_boot=N_BOOT,
                avg_block_len=best_L,
                seed=BOOTSTRAP_SEED,
                rf_daily=rf_daily,
            )

        print("Net in-sample performance metrics — AR vs CS")
        display(
            pd.concat(
                {est: boot_comp[est]["actual_metrics_net"] for est in boot_comp},
                names=["Estimator", "Strategy"],
            ).round(4)
        )

        print("Net bootstrap distribution — CAGR (AR vs CS)")
        display(
            pd.concat(
                {est: boot_comp[est]["bootstrap_tables_net"]["CAGR"] for est in boot_comp},
                names=["Estimator", "Strategy"],
            ).round(4)
        )

        print("Net bootstrap distribution — Sharpe ratio (AR vs CS)")
        display(
            pd.concat(
                {est: boot_comp[est]["bootstrap_tables_net"]["ShR"] for est in boot_comp},
                names=["Estimator", "Strategy"],
            ).round(4)
        )

        print("Net bootstrap distribution — Martin ratio (AR vs CS)")
        display(
            pd.concat(
                {est: boot_comp[est]["bootstrap_tables_net"]["Martin"] for est in boot_comp},
                names=["Estimator", "Strategy"],
            ).round(4)
        )

        print("Net bootstrap distribution — Max drawdown (AR vs CS)")
        display(
            pd.concat(
                {est: boot_comp[est]["bootstrap_tables_net"]["MxDD"] for est in boot_comp},
                names=["Estimator", "Strategy"],
            ).round(4)
        )